# 07 — Model explainability

**Objective.** Compute SHAP explanations for the deployed linear model and verify our fast implementation against the `shap` library.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (9, 4)
from ml.config import PROCESSED_DATASET, ARTIFACTS_DIR, METADATA_DIR, RAW_DIR
df = pd.read_csv(PROCESSED_DATASET)
print(f'{len(df):,} processed fact-checks')

4,137 processed fact-checks


In [2]:
import joblib, shap
from ml.explainability.explain import LinearExplainer
from ml.preprocessing.text_cleaning import model_text
clf = joblib.load(ARTIFACTS_DIR / 'classifier.joblib')
ex = LinearExplainer(clf, background_mean=np.load(ARTIFACTS_DIR / 'explainer_background_mean.npy'))
out = ex.explain(model_text('Old video from Bangladesh shared as police firing on protesters in Delhi'))
print(out['target_class'], out['method']); pd.DataFrame(out['top_features'])

MISLEADING exact_linear_shap


,feature,kind,contribution
0,video from,word,0.32381
1,old,word,0.22471
2,video,word,0.17164
3,bangladesh,word,0.10229
4,firing,word,0.09901
5,in delhi,word,0.09576
6,in,word,0.08174
7,fir,char,0.07038
8,police,word,0.06896
9,old,char,0.05219


## Additivity (SHAP efficiency)
base + Σ contributions of present features + absent-feature term = model output.

In [3]:
full = ex.explain(model_text('Viral video claims the RBI banned ₹500 notes'), top_k=10**6)
print(full['base_value'] + sum(f['contribution'] for f in full['top_features']) + full['absent_features_contribution'], '==', full['output_value'])

1.31254 == 1.31257


## Agreement with the shap library

In [4]:
feats = clf.named_steps['features']; sub = df.dropna(subset=['verdict']).head(100)
bg = feats.transform(sub.claim_text.map(model_text)); ex2 = LinearExplainer(clf).fit_background(sub.claim_text.map(model_text).tolist())
text = model_text('old video of flood shared as recent'); c = list(clf.classes_).index('OUTDATED')
lib = shap.LinearExplainer((clf.named_steps['clf'].coef_[c], clf.named_steps['clf'].intercept_[c]), bg).shap_values(feats.transform([text]))
ours = {f['feature']: f['contribution'] for f in ex2.explain(text, 'OUTDATED', top_k=8)['top_features']}
names = list(ex2.names)
pd.DataFrame({'ours': ours, 'shap library': {k: float(np.ravel(lib)[names.index(k)]) for k in ours}})

,ours,shap library
recent,0.55143,0.551427
flood,-0.18364,-0.183636
video,-0.13855,-0.138554
of,0.13803,0.138034
old,-0.13155,-0.131553
as,-0.11795,-0.117946
shared,-0.08186,-0.081863
video of,0.08183,0.081825


## Global view: strongest n-grams per class

In [5]:
W = clf.named_steps['clf'].coef_
for i, cl in enumerate(clf.classes_):
    top = np.argsort(-W[i])[:12]; print(cl, '→', ', '.join(repr(ex.names[j]) for j in top))

FALSE → 'and', 'real', 'नहीं', 'ಎಂದು', '15', 'in bengal', 'ಎಂದು ಹಂಚಿಕೊಳ್ಳಲಾಗುತ್ತಿದೆ', 'ने', 'is', 'ai', 'ಎಂಬುದು', 'has'
MISLEADING → 'video', 'ಿದೆ', 'की', 'video from', 'ವಿಡಿಯೋ ತೋರಿಸುತ್ತದೆ', 'दावा वायरल', 'ಪೊಲೀಸರನ್ನು', 'fire', 'photos show', 'people in', 'muslim', 'in'
OUTDATED → 'साल', 'ಇತ್ತೀಚಿನ', 'वीडियो वायरल', 'ವೀಡಿಯೊವನ್ನು', 'recent', 'का वीडियो', 'वीडियो', 'iaf', 'of bjp', 'after the', 'visuals show', 'of man'


## Observations
* Our sparse implementation matches `shap.LinearExplainer` to ~1e-6.
* OUTDATED is driven by temporal words (साल 'year', ಇತ್ತೀಚಿನ 'recent', 'recent'); MISLEADING by provenance phrasing ('video from', 'photos show') — plausible model behaviour, **not proof** that these words indicate misinformation.
* **Bias finding:** the word 'muslim' is among the strongest MISLEADING features. This mirrors which communal claims fact-checkers debunked (often real footage miscaptioned with a communal angle), not anything about the community; it is a reason the classifier must never judge a claim alone, and a candidate for feature masking or counterfactual evaluation.

**Conclusion.** Explanations are faithful to the model and cheap enough to compute on every request.